In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 读取噪声特征数据
df_noise = pd.read_csv(r'../数据/噪声特征.csv', encoding='utf-8-sig')
print("原始数据:")
print(df_noise.head())

原始数据:
        code  收益率标准差（波动率）        偏度         峰度   一阶自相关系数   五阶自相关系数  波动率聚集变异系数  \
0  920680.BJ     1.074350 -1.965274   3.884620 -0.846281       NaN        NaN   
1  688816.SH     0.703517 -2.235244   4.997052  0.655254       NaN        NaN   
2  688785.SH     0.700351  3.843064  14.834854  0.591124  0.070288        NaN   
3  920045.BJ     0.600236  5.503594  30.504500  0.148002 -0.011823   1.786129   
4  688795.SH     0.467533  6.917134  48.196735  0.189386 -0.053985   2.463363   

      极端值比例     噪声信号比  
0  0.000000  1.494533  
1  0.000000  2.199727  
2  0.066667  4.101409  
3  0.032258  5.340765  
4  0.020408  6.976176  


In [3]:
# 1. 特征处理
df_noise['log_kurtosis'] = np.log1p(df_noise['峰度'])

# 用于机器学习的全部噪声特征
feature_cols = [
    '收益率标准差（波动率）',
    'log_kurtosis',
    '一阶自相关系数',
    '极端值比例',
    '波动率聚集变异系数'
]
df_noise = df_noise.dropna(subset=feature_cols).copy()
# 特征标准化
scaler = MinMaxScaler()
X = df_noise[feature_cols].copy()
X = scaler.fit_transform(X)

# ---------------------------------------------------------------------------------
# 【AI 打分】随机森林自动学习噪声强度（不再人工定权重）
# ---------------------------------------------------------------------------------
y = df_noise['收益率标准差（波动率）']  # 目标：预测噪声强度
rf = RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42)
rf.fit(X, y)
df_noise['noise_score'] = rf.predict(X)  # 模型自动算出的噪声得分

# ---------------------------------------------------------------------------------
# 【AI 分组】KMeans 自动聚类分3组（不再人工三等分）
# ---------------------------------------------------------------------------------
km = KMeans(n_clusters=3, random_state=42)
df_noise['cluster'] = km.fit_predict(X)

# 自动按得分高低映射：高噪声组 / 中噪声组 / 低噪声组
cluster_mean = df_noise.groupby('cluster')['noise_score'].mean().sort_values(ascending=False)
cluster_map = {
    cluster_mean.index[0]: '高噪声组',
    cluster_mean.index[1]: '中噪声组',
    cluster_mean.index[2]: '低噪声组'
}
df_noise['group'] = df_noise['cluster'].map(cluster_map)


print("\n=== 【机器学习自动分组结果】 ===")
for g in ['高噪声组', '中噪声组', '低噪声组']:
    sub = df_noise[df_noise['group'] == g]
    print(f"\n{g} ({len(sub)}只) | 平均噪声得分: {sub['noise_score'].mean():.3f}")

print("\n=== 模型自动学习的特征重要性 ===")
for col, imp in zip(feature_cols, rf.feature_importances_):
    print(f"{col}: {imp:.1%}")

# 保存（格式不变）
df_noise[['code', 'group', 'noise_score', '收益率标准差（波动率）', 
          '峰度', '一阶自相关系数']].to_csv(r'../数据/噪声分组.csv', index=False)


=== 【机器学习自动分组结果】 ===

高噪声组 (2087只) | 平均噪声得分: 0.053

中噪声组 (2307只) | 平均噪声得分: 0.047

低噪声组 (1558只) | 平均噪声得分: 0.034

=== 模型自动学习的特征重要性 ===
收益率标准差（波动率）: 99.3%
log_kurtosis: 0.3%
一阶自相关系数: 0.1%
极端值比例: 0.1%
波动率聚集变异系数: 0.1%


In [4]:
# 根据股票的噪声特征设计滤波器参数
"""
    返回:
    - cutoff_freq: 截止频率 (归一化频率，0-0.5)
    - filter_order: 滤波器阶数
    - filter_type: 滤波器类型
    """
    # 1. 训练特征
feature_cols = [
    '收益率标准差（波动率）',
    'log_kurtosis',
    '一阶自相关系数',
    '极端值比例',
    '波动率聚集变异系数']

X = df_noise[feature_cols].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 【模型1】预测：截止频率 cutoff_freq（回归任务）
# 目标变量：用噪声得分自动生成最优目标
y_cutoff = 0.01 + 0.04 * (1 - df_noise['noise_score'] /     df_noise['noise_score'].max())

model_cutoff = RandomForestRegressor(n_estimators=100, random_state=42)
model_cutoff.fit(X_scaled, y_cutoff)
df_noise['cutoff_freq'] = model_cutoff.predict(X_scaled)
df_noise['cutoff_freq'] = df_noise['cutoff_freq'].clip(0.01, 0.06)  # 安全范围

# -----------------------------------------------------------------------------
# 【模型2】预测：滤波器阶数 filter_order（分类/整数预测）
# -----------------------------------------------------------------------------
y_order = np.clip(3 + (df_noise['峰度'] > 5).astype(int) + (df_noise['极端值比例'] > 0.01).astype(int), 3, 6)

model_order = RandomForestClassifier(n_estimators=100, random_state=42)
model_order.fit(X_scaled, y_order)
df_noise['filter_order'] = model_order.predict(X_scaled)

# -----------------------------------------------------------------------------
# 【模型3】预测：滤波器类型 filter_type（分类任务：butterworth / chebyshev）
# -----------------------------------------------------------------------------
y_type = np.where(df_noise['极端值比例'] > 0.015, 'butterworth', 'chebyshev')

model_type = RandomForestClassifier(n_estimators=100, random_state=42)
model_type.fit(X_scaled, y_type)
df_noise['filter_type'] = model_type.predict(X_scaled)

# -----------------------------------------------------------------------------
# 自动计算周期（由截止频率导出）
# -----------------------------------------------------------------------------
df_noise['cutoff_period'] = (1 / df_noise['cutoff_freq']).round().astype(int)
df_noise['cutoff_period'] = df_noise['cutoff_period'].clip(15, 60)

# ===================== 输出最终滤波器参数 =====================
filter_df = df_noise[['code', 'group', 'cutoff_freq', 'cutoff_period', 
                      'filter_order', 'filter_type']].copy()

print("=== 机器学习全自动生成的滤波器参数 ===")
print(filter_df.head(10))

# 保存
filter_df.to_csv(r'../数据/滤波器参数设置.csv', index=False, encoding='utf-8-sig')

=== 机器学习全自动生成的滤波器参数 ===
         code group  cutoff_freq  cutoff_period  filter_order  filter_type
3   920045.BJ  低噪声组     0.013941             60             5  butterworth
4   688795.SH  中噪声组     0.019371             52             5  butterworth
9   688805.SH  中噪声组     0.023014             43             5  butterworth
10  920046.BJ  中噪声组     0.023825             42             5    chebyshev
11  688796.SH  中噪声组     0.024327             41             5  butterworth
12  920985.BJ  中噪声组     0.025038             40             5    chebyshev
13  920091.BJ  中噪声组     0.025248             40             5  butterworth
14  688802.SH  中噪声组     0.026718             37             5  butterworth
15  920496.BJ  中噪声组     0.028814             35             5    chebyshev
16  920809.BJ  中噪声组     0.028475             35             5    chebyshev
